In [10]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
%matplotlib widget
from core.Log import *
import logging
from core.globals import *
from core.CardiacCTdataset import DataLoaderFactory
from core.CNNmodel import *
import pandas as pd
from core.CVsplits import *
from tqdm.notebook import tqdm
OUTER_FOLDS = 5; INNER_FOLDS = 3
from sklearn.model_selection import train_test_split
from core.modelUtils import *
setup_loggers()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#main_dataset = load_dataset_info(file="data/data_info.json")
#labels = [lbl['label'] for lbl in main_dataset]
#main_training, final_test = train_test_split(main_dataset,
#											 test_size=22,
#											 stratify=labels,
#											 random_state=67)
#print(len(final_test))
#
#for sample in main_dataset:
#	if sample in final_test: sample['pool'] = 'holdout'
#	else: sample['pool'] = 'main'

#save_dataset_info(main_dataset, file="data/data_info.json")


22


In [ ]:
#create_folds_stats(OUTER_K=5, INNER_K=3)


Generating and saving fold indices...
OUTER FOLD 0 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 1 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 2 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samples
Generating and saving fold indices...
OUTER FOLD 3 ---> 108 training, 27 evaluation samples
INNER FOLD 0 ---> 72 training, 36 validation samples
INNER FOLD 1 ---> 72 training, 36 validation samples
INNER FOLD 2 ---> 72 training, 36 validation samp

In [2]:
LR_SWEEP = [6e-4, 5e-4, 4e-4, 3e-4]
DR_SWEEP = [0.3]
TH_SWEEP = [0.4]
WD_SWEEP = [1e-4]
INNER_CV_parameters = []
ID = 1
for outer_fold_idx in range(0, 5):
	for inner_fold_idx in range(0, 3):

		# Create all combinations of hyperparameters
		for i in range(len(LR_SWEEP)):
			item = {
				"ExpID": ID,
				'OUTER_FOLD': outer_fold_idx,
				'INNER_FOLD': inner_fold_idx,
				"hypers": {
					"HPset": i+1,
					"LR": LR_SWEEP[i],
					"WD": 1e-4,
					"DR": 0.3,
					"TH": 0.4,
					"P": 5,
					"Epochs": 30,
					},
				"trained": False,
			}
			INNER_CV_parameters.append(item)
			ID += 1
print(len(INNER_CV_parameters))
save_to_json(INNER_CV_parameters, filename="training/INNER_experiments.json")
INNER_CV_parameters


60


[{'ExpID': 1,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 1,
   'LR': 0.0006,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 2,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 2,
   'LR': 0.0005,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 3,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 3,
   'LR': 0.0004,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 4,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 4,
   'LR': 0.0003,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 5,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 1,
  'hypers': {'HPset': 1,
   'LR': 0.0006,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 6,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 1,
  '

In [6]:
INNER_experiments = load_from_json("training/INNER_experiments.json")
# Filter experiments for OUTER_FOLD 0
exp_list = [item for item in INNER_experiments if item['OUTER_FOLD'] == 0]
print(len(exp_list))
exp_list


Loaded training/INNER_experiments.json.
12


[{'ExpID': 1,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 1,
   'LR': 0.0006,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 2,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 2,
   'LR': 0.0005,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 3,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 3,
   'LR': 0.0004,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 4,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 0,
  'hypers': {'HPset': 4,
   'LR': 0.0003,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 5,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 1,
  'hypers': {'HPset': 1,
   'LR': 0.0006,
   'WD': 0.0001,
   'DR': 0.3,
   'TH': 0.4,
   'P': 5,
   'Epochs': 30},
  'trained': False},
 {'ExpID': 6,
  'OUTER_FOLD': 0,
  'INNER_FOLD': 1,
  '

In [ ]:
DL = DataLoaderFactory(load_dataset(pool="main"), get_fold_stats())
pbar_experiments = tqdm(exp_list, desc=f"OUTER FOLD 0 HP SEARCH", position=0, leave=True)
for experiment in pbar_experiments:
	hypers = experiment['hypers']
	if experiment['trained'] == True: continue
	log = logging.getLogger('INNER_train')
	DR = hypers['DR']
	OUT = experiment['OUTER_FOLD']
	INN = experiment['INNER_FOLD']
	train_loader, val_loader = DL.create_inner_loaders(OUT, INN)
	model = MultiViewCNN(DR)
	best_val_loss = train_INNER_model(model, train_loader, val_loader, experiment)
	experiment['best_val_loss'] = best_val_loss
	experiment['trained'] = True
	save_to_json(exp_list, filename="training/INNER_experiments.json")
	log.info(f"--------------------------------------------------------------------------")




OUTER FOLD 0 HP SEARCH:   0%|          | 0/12 [00:00<?, ?it/s]

	↳ Experiment 1 | Training model... :   0%|          | 0/30 [00:00<?, ?it/s]

	↳ Experiment 2 | Training model... :   0%|          | 0/30 [00:00<?, ?it/s]